In [36]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
from collections import Counter

In [37]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df.head(5)

reviews_sample = reviews_df.sample(n = 10000, random_state = 42)

In [38]:
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

roberta_model = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation = True, 
    max_length = 512,
    device = device
)

Using device: CPU


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 67138.26it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
def get_roberta_sentiments(texts, batch_size = 128):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts.iloc[i:i + batch_size].tolist()
        outputs = roberta_model(batch)
        results.extend(outputs)
    return results 

In [40]:
print(reviews_sample['text'].str.len().describe())

count    10000.000000
mean       471.574200
std        438.171319
min         32.000000
25%        193.000000
50%        337.000000
75%        592.000000
max       4954.000000
Name: text, dtype: float64


In [41]:
reviews_sample = reviews_df.sample(10000, random_state = 42)
reviews_sample['text_truncated'] = reviews_sample['text'].str[:256]

roberta_results = get_roberta_sentiments(
    reviews_sample['text_truncated'],
    batch_size = 128
)


100%|██████████| 79/79 [03:39<00:00,  2.77s/it]


In [46]:
def convert_score(r):
    if r['label'] == "positive":
        return r['score']
    elif r['label'] == "neutral":
        return 0.5
    else: 
        return 1 - r['score']
reviews_sample['roberta_score'] = [convert_score(r) for r in roberta_results]


In [45]:
labels = [r['label'] for r in roberta_results]
print(Counter(labels))

Counter({'positive': 7148, 'negative': 2348, 'neutral': 504})


In [47]:
print(reviews_sample[['roberta_score']].describe())

       roberta_score
count   10000.000000
mean        0.734460
std         0.346498
min         0.033413
25%         0.500000
50%         0.960896
75%         0.984291
max         0.992579
